# Determine and Download Datasets

first we load our data from the csv into a pandas dataframe

In [112]:
import os
print(os.getcwd())

/home/lbuerger/masterthesis/master_thesis_prototype/playground/notebooks


In [113]:
import pandas as pd

df = pd.read_csv("./catalog_distributions.csv")

In [114]:
for x in df.distributions[:20]:
    print(len(x))

316
318
2347
2347
3990
3990
4010
776
317
332
325
1137
289
291
271
378
360
374
617
1586


In [115]:
import ast

print(list(df.distributions)[0])

[{'uri': 'https://ckan.govdata.de/dataset/d7ffe65c-c473-46a9-8a03-98bf3fba57e4/resource/af104411-a1b1-4eb4-a96c-c6632e6fbcca', 'title': 'Dynamisches Angebot der Taxi Dortmund eG', 'access_url': 'https://mobilithek.info//offers/636910977297584128#accessURL', 'download_url': None, 'format': None, 'media_type': None}]


in one run, pandas recognized the distributions column as a string, so we need to convert it back to a list

In [116]:
df['distributions'] = df['distributions'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])

we now iterate over all distributions and count the media types to get a feeling what types exist and which of them are tabular

In [117]:
from collections import Counter

media_type_counter = Counter()


def tabular(x):
    if not isinstance(x, list):
        return

    for item in x:
        if not isinstance(item, dict):
            continue

        if item.get("media_type"):
            media_type_counter[item["media_type"]] += 1


df['distributions'].apply(tabular)

print("Media-Type Distribution:")
for media_type, count in media_type_counter.most_common():
    print(f"{count}: {media_type}")

Media-Type Distribution:
135915: application/x-netcdf
52411: https://www.iana.org/assignments/media-types/text/csv
34167: text/csv
26312: application/pdf
25612: application/zip
14814: https://www.iana.org/assignments/media-types/text/xml
11902: https://www.iana.org/assignments/media-types/text/xlsx
10274: text/plain
7826: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
6490: application/xml
6220: https://www.iana.org/assignments/media-types/application/pdf
4947: https://www.iana.org/assignments/media-types/application/json
4120: https://www.iana.org/assignments/media-types/application/zip
2839: https://www.iana.org/assignments/media-types/application/geo+json
2163: application/json
2140: text/html
2125: application/vnd.ms-excel
1927: text/csv+extended
1649: https://www.iana.org/assignments/media-types/application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
958: image/tiff
925: https://www.iana.org/assignments/media-types/application/gml+xml
905: https://www.

we pick the tabular media types by hand and save them as constants in a list

In [118]:
tabular_media_types = [
    "https://www.iana.org/assignments/media-types/text/csv",
    "text/csv",
    "https://www.iana.org/assignments/media-types/text/xlsx",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    "application/vnd.ms-excel",
    "text/csv+extended",
    "https://www.iana.org/assignments/media-types/application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    "https://www.iana.org/assignments/media-types/application/vnd.ms-excel",
    "https://www.iana.org/assignments/media-types/application/vnd.oasis.opendocument.spreadsheet",
    "application/vnd.oasis.opendocument.spreadsheet",
    "text/tab-separated-values",
    "application/xlsx",
    "https://www.iana.org/assignments/media-types/application/json/application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
]

fot the study / experiment, we start with csv datasets

In [119]:
csv_media_types = [
    "https://www.iana.org/assignments/media-types/text/csv",
    "text/csv",
    "text/csv+extended"
]

In [120]:
def has_tabular(distributions):
    for dist in distributions:
        if not isinstance(dist, dict):
            continue
        media_type = dist.get("media_type")
        if media_type in tabular_media_types:
            return True
    return False


df['has_tabular'] = df['distributions'].apply(has_tabular)
df["has_tabular"]

0         False
1         False
2         False
3         False
4         False
          ...  
147254    False
147255    False
147256    False
147257    False
147258    False
Name: has_tabular, Length: 147259, dtype: bool

In [121]:
def has_csv(distributions):
    for dist in distributions:
        if not isinstance(dist, dict):
            continue
        media_type = dist.get("media_type")
        if media_type in csv_media_types:
            return True
    return False

df['has_csv'] = df['distributions'].apply(has_csv)
df["has_csv"]

0         False
1         False
2         False
3         False
4         False
          ...  
147254    False
147255    False
147256    False
147257    False
147258    False
Name: has_csv, Length: 147259, dtype: bool

query all datasets that contain csv distributions

In [122]:
df.query("has_csv == True")

,title,rdf_about,distributions,has_tabular,has_csv
85,Detektierte Erschütterungen an der LKW-Auflieg...,https://mobilithek.info//offers/57321669459012...,[{'uri': 'https://ckan.govdata.de/dataset/fd05...,True,True
127,Dauerzählstellen (Rad) Hamburg,https://mobilithek.info//offers/-3869360234918...,[{'uri': 'https://ckan.govdata.de/dataset/19a4...,True,True
128,Dauerzählstellen (Rad) Hamburg,https://mobilithek.info//offers/-8196963435227...,[{'uri': 'https://ckan.govdata.de/dataset/0fe5...,True,True
132,Datengrundlage zur Stoppzeitanalyse auf der le...,https://mobilithek.info//offers/93810129412002...,[{'uri': 'https://ckan.govdata.de/dataset/25f2...,True,True
134,Daten zur Evaluation des Leezenflow-Systems in...,https://opendata.stadt-muenster.de/dataset/dat...,[{'uri': 'https://opendata.stadt-muenster.de/d...,True,True
...,...,...,...,...,...
145163,Ansprechpartnerinnen zur Vergabe einer BNR-ZD ...,https://service.brandenburg.de/service/de/adre...,[{'uri': 'https://service.brandenburg.de/servi...,True,True
145164,Forschungseinrichtungen im Agrarbereich,https://service.brandenburg.de/service/de/adre...,[{'uri': 'https://service.brandenburg.de/servi...,True,True
145217,Ansprechstellen des Zolls für die Bekämpfung v...,https://ckan.govdata.de/dataset/308a7cde-23e9-...,[{'uri': 'https://ckan.govdata.de/dataset/308a...,True,True
145218,Ansprechstellen des Zolls für die Festsetzung ...,https://ckan.govdata.de/dataset/ef42d9fa-5710-...,[{'uri': 'https://ckan.govdata.de/dataset/ef42...,True,True


from those datasets we now pick 150 randomly for the study (3x50 to account for failures)

In [123]:
df_csv_samples = df.query("has_csv == True").sample(75, random_state=67)

df_csv_samples

,title,rdf_about,distributions,has_tabular,has_csv
144193,"Pflegebedürftige (stationär): Deutschland, Sti...",https://www-genesis.destatis.de/genesis-old/do...,[{'uri': 'https://www-genesis.destatis.de/gene...,True,True
62138,Anzahl der genehmigten Wohnungen insgesamt (Ne...,https://region.statistik-nord.de/detail_timeli...,[{'uri': 'https://region.statistik-nord.de/det...,True,True
13444,Vogtareuth: Tourismus,https://open.bydata.de/api/hub/repo/datasets/4...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True
12655,Prebitz: Tourismus,https://open.bydata.de/api/hub/repo/datasets/4...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True
83413,"Verunglückte: Deutschland, Jahre, Geschlecht, ...",https://www-genesis.destatis.de/genesis-old/do...,[{'uri': 'https://www-genesis.destatis.de/gene...,True,True
...,...,...,...,...,...
36435,D15 Bundestagswahl Niederzier,https://offenedaten.kdvz.nrw/dataset/d15-bunde...,[{'uri': 'https://offenedaten.kdvz.nrw/dataset...,True,True
119884,Betriebe im Bauhauptgewerbe nach WZ-2-Stellern...,https://region.statistik-nord.de/detail_timeli...,[{'uri': 'https://region.statistik-nord.de/det...,True,True
16082,Berchtesgadener Land (Lkr): Fläche,https://open.bydata.de/api/hub/repo/datasets/1...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True
74095,"Außenhandel: Einfuhr, Ausfuhr, Kontinente (Gli...",https://www.statistik.sachsen.de/genonline/onl...,[{'uri': 'https://www.statistik.sachsen.de/gen...,True,True


now we have to randomly choose one csv distribution per dataset to download. this is only needed if there are multiple csv distributions per dataset

In [124]:
import numpy as np

np.random.seed(67)

def select_random_csv(distributions):
    if not isinstance(distributions, list):
        return None

    csv_distributions = [
        item for item in distributions
        if isinstance(item, dict) and item.get("media_type") and "csv" in item["media_type"].lower()
    ]

    if not csv_distributions:
        return None

    return csv_distributions[np.random.randint(len(csv_distributions))]

df_csv_samples["study_distribution"] = df_csv_samples["distributions"].apply(select_random_csv)

In [125]:
df_csv_samples["study_distribution"]

144193    {'uri': 'https://www-genesis.destatis.de/genes...
62138     {'uri': 'https://region.statistik-nord.de/deta...
13444     {'uri': 'https://open.bydata.de/api/hub/repo/d...
12655     {'uri': 'https://open.bydata.de/api/hub/repo/d...
83413     {'uri': 'https://www-genesis.destatis.de/genes...
                                ...                        
36435     {'uri': 'https://offenedaten.kdvz.nrw/dataset/...
119884    {'uri': 'https://region.statistik-nord.de/deta...
16082     {'uri': 'https://open.bydata.de/api/hub/repo/d...
74095     {'uri': 'https://www.statistik.sachsen.de/geno...
119642    {'uri': 'https://region.statistik-nord.de/deta...
Name: study_distribution, Length: 75, dtype: object

In [126]:
df_csv_samples.to_csv("../../playground/example_datasets.csv", index=True)

let's extract the download urls for the selected distributions

In [127]:
df_csv_samples["download_url"] = df_csv_samples["study_distribution"].apply(
    lambda x: x.get("access_url") if isinstance(x, dict) else None
)

In [128]:
df_csv_samples["download_url"]
type(df_csv_samples["download_url"])

pandas.Series

next step is to generate the dataset title, like it's done in CKAN, to create the rdf metadata url for downloading the metadata rdf file

In [129]:
import re

def munge_title_to_name(title, max_length=91):
    """
    Munge a title into a name suitable for use in a URL.
    Based on CKAN's munge_title_to_name function.
    """
    if not title:
        return ""

    title = str(title)

    # Convert to lowercase
    title = title.lower()

    # Replace German umlauts and special characters
    title = title.replace('ä', 'a')
    title = title.replace('ö', 'o')
    title = title.replace('ü', 'u')
    title = title.replace('ß', 'ss')
    title = title.replace('Ä', 'a')
    title = title.replace('Ö', 'o')
    title = title.replace('Ü', 'u')

    # Replace non-alphanumeric characters (except hyphens, underscores) with spaces
    title = re.sub(r'[^\w\s-]', '', title)

    # Replace spaces and underscores with hyphens
    title = re.sub(r'[\s_]+', '-', title)

    # Collapse multiple hyphens
    title = re.sub('-+', '-', title)

    title = title.rstrip('-')

    # Limit length
    if len(title) > max_length:
        title = title[:max_length]


    return title

# Apply to dataframe
df_csv_samples["url_slug"] = df_csv_samples["title"].apply(
    lambda x: munge_title_to_name(x, max_length=95)
)

In [130]:
df_csv_samples["url_slug"]

144193    pflegebedurftige-stationar-deutschland-stichta...
62138     anzahl-der-genehmigten-wohnungen-insgesamt-neu...
13444                                  vogtareuth-tourismus
12655                                     prebitz-tourismus
83413     verungluckte-deutschland-jahre-geschlecht-alte...
                                ...                        
36435                         d15-bundestagswahl-niederzier
119884    betriebe-im-bauhauptgewerbe-nach-wz-2-stellern...
16082                       berchtesgadener-land-lkr-flache
74095     aussenhandel-einfuhr-ausfuhr-kontinente-gliede...
119642    betriebe-im-bauhauptgewerbe-nach-wz-2-stellern...
Name: url_slug, Length: 75, dtype: str

In [131]:
base_url = "https://ckan.govdata.de/dataset/"

df_csv_samples["rdf_url"] = (
    base_url +
    df_csv_samples["url_slug"] +
    ".rdf"
)

df_csv_samples["rdf_url"]

144193    https://ckan.govdata.de/dataset/pflegebedurfti...
62138     https://ckan.govdata.de/dataset/anzahl-der-gen...
13444     https://ckan.govdata.de/dataset/vogtareuth-tou...
12655     https://ckan.govdata.de/dataset/prebitz-touris...
83413     https://ckan.govdata.de/dataset/verungluckte-d...
                                ...                        
36435     https://ckan.govdata.de/dataset/d15-bundestags...
119884    https://ckan.govdata.de/dataset/betriebe-im-ba...
16082     https://ckan.govdata.de/dataset/berchtesgadene...
74095     https://ckan.govdata.de/dataset/aussenhandel-e...
119642    https://ckan.govdata.de/dataset/betriebe-im-ba...
Name: rdf_url, Length: 75, dtype: str

we got the dataset download url and the metadata url.

now we create for each dataset a folder and download the csv file into that folder and its corresponding metadata rdf file

In [132]:
import os
import requests
from pathlib import Path

output_dir = Path("../../data")
output_dir.mkdir(exist_ok=True)

session = requests.Session()

def download_file(session, url, filepath, label):
    try:
        response = session.get(url, timeout=90)
        response.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(response.content)
        print(f"{label} ✓")
        return True
    except Exception as e:
        print(f"{label} ✗ - {e}")
        return False

successful_pairs = []
failed_indices = []

# Try up to 3 rounds to get 50 successful pairs
for round_num in range(1, 4):
    print(f"\n=== Round {round_num} ===")
    
    for idx, row in df_csv_samples.iterrows():
        # Skip already successful pairs
        if idx in successful_pairs:
            continue
            
        row_dir = Path(os.path.join(output_dir, str(idx)))
        row_dir.mkdir(exist_ok=True)

        # Download CSV
        csv_success = download_file(
            session,
            row.get("download_url"),
            os.path.join(row_dir, "dataset.csv"),
            f"Row {idx}: CSV"
        )

        # Download RDF
        rdf_success = download_file(
            session,
            row.get("rdf_url"),
            os.path.join(row_dir, "metadata.rdf"),
            f"Row {idx}: RDF"
        )

        # If both succeeded, count as successful pair
        if csv_success and rdf_success:
            successful_pairs.append(idx)
            print(f"  >>> Successful pair! Total: {len(successful_pairs)}/25")
        else:
            failed_indices.append(idx)
            # Clean up failed folder
            if row_dir.exists() and any(row_dir.iterdir()):
                for f in row_dir.iterdir():
                    f.unlink()
                row_dir.rmdir()
        
        # Stop if we have 25 successful pairs
        if len(successful_pairs) >= 25:
            print(f"\nReached 25 successful pairs!")
            break
    
    if len(successful_pairs) >= 25:
        break

print(f"\n=== Summary ===")
print(f"Successful pairs: {len(successful_pairs)}")
print(f"Failed indices: {len(failed_indices)}")

session.close()


=== Round 1 ===
Row 144193: CSV ✗ - 403 Client Error: Forbidden for url: https://www-genesis.destatis.de/genesis-old/downloads/00/tables/22412-0004_00.csv
Row 144193: RDF ✗ - 404 Client Error: NOT FOUND for url: https://ckan.govdata.de/dataset/pflegebedurftige-stationar-deutschland-stichtaggeschlecht-altersgruppen.rdf
Row 62138: CSV ✓
Row 62138: RDF ✗ - 404 Client Error: NOT FOUND for url: https://ckan.govdata.de/dataset/anzahl-der-genehmigten-wohnungen-insgesamt-neubau-und-baumassnahmen-in-kohn.rdf
Row 13444: CSV ✓
Row 13444: RDF ✓
  >>> Successful pair! Total: 1/25
Row 12655: CSV ✓
Row 12655: RDF ✓
  >>> Successful pair! Total: 2/25
Row 83413: CSV ✗ - 403 Client Error: Forbidden for url: https://www-genesis.destatis.de/genesis-old/downloads/00/tables/46241-0007_00.csv
Row 83413: RDF ✗ - 404 Client Error: NOT FOUND for url: https://ckan.govdata.de/dataset/verungluckte-deutschland-jahre-geschlecht-altersgruppenart-der-verkehrsbeteiligung-ortslage-sch.rdf
Row 60145: CSV ✓
Row 60145: RD

we tried to retrie all data, for the datasets with rdf we try to handle it by hand, its probably a url generation error

In [133]:
df_csv_samples

,title,rdf_about,distributions,has_tabular,has_csv,study_distribution,download_url,url_slug,rdf_url
144193,"Pflegebedürftige (stationär): Deutschland, Sti...",https://www-genesis.destatis.de/genesis-old/do...,[{'uri': 'https://www-genesis.destatis.de/gene...,True,True,{'uri': 'https://www-genesis.destatis.de/genes...,https://www-genesis.destatis.de/genesis-old/do...,pflegebedurftige-stationar-deutschland-stichta...,https://ckan.govdata.de/dataset/pflegebedurfti...
62138,Anzahl der genehmigten Wohnungen insgesamt (Ne...,https://region.statistik-nord.de/detail_timeli...,[{'uri': 'https://region.statistik-nord.de/det...,True,True,{'uri': 'https://region.statistik-nord.de/deta...,https://region.statistik-nord.de/detail_timeli...,anzahl-der-genehmigten-wohnungen-insgesamt-neu...,https://ckan.govdata.de/dataset/anzahl-der-gen...
13444,Vogtareuth: Tourismus,https://open.bydata.de/api/hub/repo/datasets/4...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True,{'uri': 'https://open.bydata.de/api/hub/repo/d...,https://www.statistikdaten.bayern.de/genesis/o...,vogtareuth-tourismus,https://ckan.govdata.de/dataset/vogtareuth-tou...
12655,Prebitz: Tourismus,https://open.bydata.de/api/hub/repo/datasets/4...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True,{'uri': 'https://open.bydata.de/api/hub/repo/d...,https://www.statistikdaten.bayern.de/genesis/o...,prebitz-tourismus,https://ckan.govdata.de/dataset/prebitz-touris...
83413,"Verunglückte: Deutschland, Jahre, Geschlecht, ...",https://www-genesis.destatis.de/genesis-old/do...,[{'uri': 'https://www-genesis.destatis.de/gene...,True,True,{'uri': 'https://www-genesis.destatis.de/genes...,https://www-genesis.destatis.de/genesis-old/do...,verungluckte-deutschland-jahre-geschlecht-alte...,https://ckan.govdata.de/dataset/verungluckte-d...
...,...,...,...,...,...,...,...,...,...
36435,D15 Bundestagswahl Niederzier,https://offenedaten.kdvz.nrw/dataset/d15-bunde...,[{'uri': 'https://offenedaten.kdvz.nrw/dataset...,True,True,{'uri': 'https://offenedaten.kdvz.nrw/dataset/...,https://offenedaten.kdvz.nrw/dataset/d15-bunde...,d15-bundestagswahl-niederzier,https://ckan.govdata.de/dataset/d15-bundestags...
119884,Betriebe im Bauhauptgewerbe nach WZ-2-Stellern...,https://region.statistik-nord.de/detail_timeli...,[{'uri': 'https://region.statistik-nord.de/det...,True,True,{'uri': 'https://region.statistik-nord.de/deta...,https://region.statistik-nord.de/detail_timeli...,betriebe-im-bauhauptgewerbe-nach-wz-2-stellern...,https://ckan.govdata.de/dataset/betriebe-im-ba...
16082,Berchtesgadener Land (Lkr): Fläche,https://open.bydata.de/api/hub/repo/datasets/1...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True,{'uri': 'https://open.bydata.de/api/hub/repo/d...,https://www.statistikdaten.bayern.de/genesis/o...,berchtesgadener-land-lkr-flache,https://ckan.govdata.de/dataset/berchtesgadene...
74095,"Außenhandel: Einfuhr, Ausfuhr, Kontinente (Gli...",https://www.statistik.sachsen.de/genonline/onl...,[{'uri': 'https://www.statistik.sachsen.de/gen...,True,True,{'uri': 'https://www.statistik.sachsen.de/geno...,https://www.statistik.sachsen.de/genonline/onl...,aussenhandel-einfuhr-ausfuhr-kontinente-gliede...,https://ckan.govdata.de/dataset/aussenhandel-e...
